# CS781 Capstone — Phase 1: Legal Language Model Pretraining

This notebook trains a **from-scratch** legal language model on the official Phase-1 corpus
(`Exploration-Lab/CS-781-Capstone`) and produces a Codabench-ready `submission.zip`.

**How to use this notebook for your 4 required Phase-1 architecture submissions**

The competition requires you to explore **at least four substantially different architectures**
(Sec. 7.3 of the capstone doc), and FAQ #5 confirms these four types specifically:
`decoder-only`, `encoder-decoder`, `encoder/hybrid`, and `lightweight/efficient`.

This single notebook implements all four, **fully from scratch** (no Hugging Face model
classes, no pretrained weights — only `torch.nn` building blocks), each in its own
architecturally distinct way:

| `ARCHITECTURE` value | Design | What makes it "substantially different" |
|---|---|---|
| `decoder_only` | Classic GPT-style decoder | Learned absolute position embeddings, LayerNorm, GELU MLP, standard multi-head self-attention |
| `encoder_decoder` | T5-style encoder-decoder | Separate encoder/decoder stacks, **relative position bias** (no absolute positions), cross-attention |
| `hybrid_prefixlm` | UniLM-style single-stack hybrid | One Transformer stack but a **prefix-bidirectional + causal-continuation attention mask**, RoPE, RMSNorm, SwiGLU |
| `lightweight` | Llama-style efficient decoder | RoPE, RMSNorm, SwiGLU, **Grouped-Query Attention** (fewer KV heads → smaller/faster) |

**Workflow:** set `ARCHITECTURE` in the CONFIG cell below to one of the four values, run the
whole notebook top to bottom, download `submission.zip`, and submit to Codabench. Repeat for
the other three values (that's your 4 required submissions — do more for extra credit, e.g.
different data mixes or hyperparameters per architecture).

**This notebook runs unchanged on Kaggle, Google Colab, or a plain local Jupyter server.** The
login/storage cell (Section 3) auto-detects which one you're on and adapts how it reads your
Hugging Face token and where it saves files -- there is nothing platform-specific to edit.

**Important — this code has not been executed.** It is written to be correct and to follow the
capstone spec closely, but you should first do a *smoke test* with tiny values
(`TRAIN_TOKENS` ~ 2,000,000, `VAL_TOKENS` ~ 100,000, `MAX_STEPS` ~ 50) to confirm the
whole pipeline runs end-to-end before committing a full multi-hour training run.

**Rules this notebook respects (Sec. 7.5–7.6):**
- Trains from scratch, no pretrained weights, no closed APIs.
- Keeps every architecture's parameter count between 50M and 300M (asserted in code).
- Uses only the four official Phase-1 train files.
- Target hardware: a single free Kaggle/Colab GPU (P100/T4, ~16GB).

## 1. Install dependencies

In [ ]:
!pip -q install -U tokenizers transformers datasets huggingface_hub accelerate evaluate bert_score rouge-score sacrebleu tqdm ijson

## 2. Imports

In [ ]:
import os
import json
import math
import time
import random
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)

## 3. Hugging Face login and storage setup

This cell auto-detects whether it's running on **Kaggle**, **Google Colab**, or something else,
and adapts accordingly:

- **On Kaggle:** add a Kaggle secret named `HF` (Add-ons -> Secrets in the notebook editor)
  containing your Hugging Face access token, and enable it for this notebook.
- **On Colab:** add a Colab secret named `HF` (click the key icon in the left sidebar)
  containing your token, and grant this notebook access when prompted.
- **Anywhere else** (or if no secret named `HF` is found): you'll be prompted to paste your
  token via a hidden `getpass` input instead.

It also picks a writable base directory for the tokenizer/checkpoints/predictions: Kaggle's
`/kaggle/working`, or on Colab, **your Google Drive** (mounted below) so training checkpoints
survive a Colab disconnect -- set `USE_DRIVE = False` below to keep everything on Colab's
ephemeral local disk instead (faster, but lost if the runtime recycles).

In [ ]:
import getpass


def _detect_environment():
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    try:
        import kaggle_secrets  # noqa: F401
        return "kaggle"
    except ImportError:
        pass
    return "other"


ENVIRONMENT = _detect_environment()
print("Detected environment:", ENVIRONMENT)

# --- Writable base directory (survives across cells / reruns of this notebook) ---
USE_DRIVE = True   # Colab only: mount Google Drive so checkpoints survive a disconnect

if ENVIRONMENT == "kaggle":
    BASE_DIR = "/kaggle/working"
elif ENVIRONMENT == "colab":
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = "/content/drive/MyDrive"
    else:
        BASE_DIR = "/content"
else:
    BASE_DIR = os.getcwd()

PROJECT_DIR = os.path.join(BASE_DIR, "cs781_phase1")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project directory:", PROJECT_DIR)

# Redirect Hugging Face's own download cache into the same persistent storage *before*
# huggingface_hub is imported anywhere -- it reads HF_HOME into module-level constants at
# import time, so setting this any later would silently have no effect. By default the cache
# lives under ~/.cache/huggingface on the VM's local (ephemeral) disk, so on Colab a runtime
# disconnect would otherwise force you to re-download the multi-GB train files from scratch on
# every one of your 4 required architecture reruns.
os.environ["HF_HOME"] = os.path.join(PROJECT_DIR, "hf_cache")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print("Hugging Face cache directory:", os.environ["HF_HOME"])

# --- Hugging Face token (import deferred until after HF_HOME is set, see above) ---
from huggingface_hub import login

HF_TOKEN = None
if ENVIRONMENT == "kaggle":
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
elif ENVIRONMENT == "colab":
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("No HF secret found -- paste your Hugging Face access token (hf_...): ")

os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

## 4. GPU check

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 5. Master configuration

**This is the only cell you need to change between your four required runs.**
Set `ARCHITECTURE` to one of `"decoder_only"`, `"encoder_decoder"`, `"hybrid_prefixlm"`,
`"lightweight"`, then run the whole notebook.

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ======================================================================
# >>> CHANGE THIS between runs to produce your 4 required submissions <<<
# One of: "decoder_only", "encoder_decoder", "hybrid_prefixlm", "lightweight"
ARCHITECTURE = "decoder_only"
# ======================================================================

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

DATA_FILES = {
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
}

# The capstone doc (Sec. 7.2) shows this file as "Phase1- Test.json"; some mirrors of the
# starter code use "test.json". We try both and use whichever exists on the HF repo.
TEST_FILE_CANDIDATES = [
    "Phase_1_Pretraining/Test/Phase1- Test.json",
    "Phase_1_Pretraining/Test/test.json",
]

VOCAB_SIZE   = 32000          # BPE vocab size -- another experiment axis (Sec. 7.4)
TRAIN_TOKENS = 120_000_000    # lower this for a quick smoke test, raise it if your GPU quota allows
VAL_TOKENS   = 3_000_000
BLOCK_SIZE   = 1024           # causal-LM training context length
PROMPT_WORDS = 512            # matches the official test prompt length (Sec. 7.2.4)
CONT_WORDS   = 128            # matches the official expected continuation length (Sec. 7.2.5)

# How much of the corpus to pull from each source, as a fraction of TRAIN_TOKENS+VAL_TOKENS.
# This is itself one of the experiment axes suggested in Sec. 7.4 ("Data selection and
# mixing") -- vary these ratios across your additional (beyond the required 4) submissions.
#
# Note: there is deliberately no "number of documents per source" setting to pick. The
# official train files are huge (the capstone doc puts the Supreme Court file alone at
# roughly 600-700M tokens; the starter-code README confirms this), and nobody knows how
# many documents that corresponds to without inspecting the live data first. Instead, the
# cell below streams each source file and stops pulling from it once it has collected
# enough *raw words* to comfortably cover that source's share of TRAIN_TOKENS+VAL_TOKENS --
# so the only number you actually have to choose remains TRAIN_TOKENS itself.
SOURCE_MIX = {"acts": 0.15, "high_courts": 0.35, "supreme_court": 0.35, "tribunals": 0.15}
assert abs(sum(SOURCE_MIX.values()) - 1.0) < 1e-6, "SOURCE_MIX ratios must sum to 1.0"

AVG_TOKENS_PER_WORD = 1.3      # rough BPE heuristic for English/legal text -- only used to
                                # size the raw-text pull below; build_token_stream() still
                                # enforces the exact TRAIN_TOKENS/VAL_TOKENS budget later
                                # using real token counts, so this heuristic being off by a
                                # bit is harmless.
PULL_SAFETY_MARGIN = 1.3       # pull ~30% more raw words per source than the estimate needs,
                                # as a buffer against the heuristic above being imprecise

WORD_BUDGET_PER_SOURCE = {
    name: int((TRAIN_TOKENS + VAL_TOKENS) * ratio / AVG_TOKENS_PER_WORD * PULL_SAFETY_MARGIN)
    for name, ratio in SOURCE_MIX.items()
}

EPOCHS            = 1
BATCH_SIZE        = 8
GRAD_ACCUM_STEPS  = 4
LR                = 3e-4
WEIGHT_DECAY      = 0.1
WARMUP_STEPS      = 500
MAX_STEPS         = None      # set an int to hard-cap steps (useful given Kaggle/Colab session time limits)
EVAL_EVERY        = 500
LOG_EVERY         = 50
SAVE_EVERY        = 1000

OUTPUT_DIR = os.path.join(PROJECT_DIR, f"run_{ARCHITECTURE}")
CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

# The tokenizer is shared/reused across all four architecture runs for comparability.
TOKENIZER_DIR = os.path.join(PROJECT_DIR, "bpe_tokenizer")

PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

# Per-architecture model configs. The hard competition limit is 50M-300M parameters, but the
# starter-code README notes 100M-150M is "strongly recommended" -- these configs target that
# sweet spot (printed and asserted later in the notebook). Do not just change these numbers
# without re-checking the printed parameter count.
MODEL_CONFIGS = {
    "decoder_only":    dict(n_layer=12, n_embd=768, n_head=12, d_ff=3072),
    "encoder_decoder": dict(n_enc_layer=6, n_dec_layer=6, d_model=768, n_head=8, d_ff=3072),
    "hybrid_prefixlm": dict(n_layer=12, d_model=640, n_head=10, d_ff=2560),
    "lightweight":     dict(n_layer=12, d_model=768, n_head=12, n_kv_head=3, d_ff=2176),
}

assert ARCHITECTURE in MODEL_CONFIGS, f"Unknown ARCHITECTURE: {ARCHITECTURE}"
print("Architecture selected:", ARCHITECTURE)
print("Model config:", MODEL_CONFIGS[ARCHITECTURE])
print("Output dir:", OUTPUT_DIR)
print("Approximate word budget per source:", {k: f"{v:,}" for k, v in WORD_BUDGET_PER_SOURCE.items()})

## 6. Download and load the Phase-1 training corpus

In [ ]:
import ijson
from huggingface_hub import hf_hub_download


def download_first_available(filename_candidates, **kwargs):
    last_err = None
    for fn in filename_candidates:
        try:
            path = hf_hub_download(filename=fn, **kwargs)
            return path, fn
        except Exception as e:
            last_err = e
    raise last_err


def load_json_records(filename):
    """Full json.load() -- only safe for small files (e.g. the test-prompt file).
    Do NOT use this for the Phase-1 train files: they can be 10+ GB and a plain
    json.load() on a file that size is exactly what triggers a Kaggle/Colab
    out-of-memory kernel restart."""
    path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        data = list(data.values())
    return data


def stream_json_records(path, max_records=None):
    """Stream records out of a (possibly multi-GB) JSON file without ever
    materializing the whole file as Python objects. Reads incrementally with
    `ijson` and stops as soon as `max_records` items have been yielded, so
    peak memory depends only on how many documents you actually keep -- not
    on the file's size on disk. Handles both a top-level JSON array of
    records (`[{...}, {...}]`) and a top-level JSON object keyed by id
    (`{"1": {...}, "2": {...}}`)."""
    count = 0
    with open(path, "rb") as f:
        first_byte = f.read(1)
        while first_byte.isspace():
            first_byte = f.read(1)
        f.seek(0)
        items = ijson.items(f, "item") if first_byte == b"[" else (v for _, v in ijson.kvitems(f, ""))
        for item in items:
            yield item
            count += 1
            if max_records is not None and count >= max_records:
                return


print("Downloading Phase-1 training sources from Hugging Face "
      "(large files -- first run can take a while; subsequent runs reuse the HF cache)...")
raw_documents = []
for source_name, filename in tqdm(DATA_FILES.items(), desc="sources", total=len(DATA_FILES)):
    local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)
    word_budget = WORD_BUDGET_PER_SOURCE[source_name]
    taken, words_collected = 0, 0
    pbar = tqdm(stream_json_records(local_path, max_records=None), desc=f"  {source_name}", unit="doc", leave=False)
    for r in pbar:
        text = r.get("text", "") if isinstance(r, dict) else str(r)
        if text and text.strip():
            raw_documents.append({"source": source_name, "text": text})
            taken += 1
            words_collected += len(text.split())
            pbar.set_postfix(docs=taken, words=f"{words_collected:,}/{word_budget:,}")
            if words_collected >= word_budget:
                break
    pbar.close()
    print(f"  {source_name}: using {taken} documents (~{words_collected:,} words)")

random.shuffle(raw_documents)
print("Total documents loaded:", len(raw_documents))

## 7. Train (or reuse) a BPE tokenizer on the legal corpus

Per Sec. 7.3: "You should use the BPE tokenizer trained for the course assignment, or you may
train another tokenizer specifically for this phase." This trains a fresh ByteLevel BPE
tokenizer on a sample of the legal corpus with the `tokenizers` library. If you already have a
trained BPE tokenizer from an earlier assignment, you can swap this cell out for loading it --
just make sure it exposes `encode(text) -> ids` and `decode(ids) -> text`.

In [ ]:
from tokenizers import ByteLevelBPETokenizer

os.makedirs(TOKENIZER_DIR, exist_ok=True)
tokenizer_already_trained = os.path.exists(os.path.join(TOKENIZER_DIR, "vocab.json"))

if not tokenizer_already_trained:
    print("Training a ByteLevel BPE tokenizer on the legal corpus...")
    corpus_txt_path = os.path.join(PROJECT_DIR, "_tokenizer_corpus.txt")
    with open(corpus_txt_path, "w", encoding="utf-8") as f:
        for doc in tqdm(raw_documents[:20000], desc="writing tokenizer corpus"):   # a subsample is enough to learn a good BPE vocab
            f.write(doc["text"].replace("\n", " ") + "\n")

    bpe_trainer = ByteLevelBPETokenizer()
    bpe_trainer.train(
        files=[corpus_txt_path],
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=["<pad>", "<bos>", "<eos>", "<unk>", "<sep>"],
    )
    bpe_trainer.save_model(TOKENIZER_DIR)
    print("Tokenizer trained and saved to:", TOKENIZER_DIR)
else:
    print("Reusing previously trained tokenizer in:", TOKENIZER_DIR)

tok = ByteLevelBPETokenizer(
    os.path.join(TOKENIZER_DIR, "vocab.json"),
    os.path.join(TOKENIZER_DIR, "merges.txt"),
)

PAD_ID, BOS_ID, EOS_ID, UNK_ID, SEP_ID = 0, 1, 2, 3, 4
VOCAB_SIZE_ACTUAL = tok.get_vocab_size()


def encode(text):
    return tok.encode(text).ids


def decode(ids):
    return tok.decode(ids)


print("Actual vocabulary size:", VOCAB_SIZE_ACTUAL)
_sample_ids = encode(raw_documents[0]["text"][:200])
print("Sample ids:", _sample_ids[:20], "...")
print("Round-trip decode:", decode(_sample_ids)[:200])

## 8. Dataset builders

Two dataset "shapes" are needed:
- **Contiguous token blocks** for standard causal-LM training (`decoder_only`, `lightweight`).
- **(prompt, continuation) pairs** built with a sliding window matching the official
  512-word-prompt / 128-word-continuation test format, for `encoder_decoder` and
  `hybrid_prefixlm`.

In [ ]:
def build_token_stream(documents, token_budget):
    stream = []
    pbar = tqdm(documents, desc="tokenizing corpus", unit="doc")
    for doc in pbar:
        ids = encode(doc["text"])
        stream.extend(ids)
        stream.append(EOS_ID)
        pbar.set_postfix(tokens=f"{len(stream):,}/{token_budget:,}")
        if len(stream) >= token_budget:
            break
    pbar.close()
    return stream[:token_budget]


def make_blocks(tokens, block_size):
    n = (len(tokens) // block_size) * block_size
    tokens = tokens[:n]
    return [tokens[i:i + block_size] for i in range(0, n, block_size)]


class BlockLMDataset(Dataset):
    def __init__(self, blocks):
        self.blocks = blocks

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, idx):
        return torch.tensor(self.blocks[idx], dtype=torch.long)


def collate_blocks(batch):
    return {"input_ids": torch.stack(batch, dim=0)}

In [ ]:
def build_prefix_continuation_pairs(documents, prompt_words=None, cont_words=None,
                                     stride_words=None, max_pairs=200000):
    prompt_words = prompt_words or PROMPT_WORDS
    cont_words = cont_words or CONT_WORDS
    window = prompt_words + cont_words
    stride_words = stride_words or window
    pairs = []
    pbar = tqdm(documents, desc="building prompt/continuation pairs", unit="doc")
    for doc in pbar:
        words = doc["text"].split()
        if len(words) < window:
            continue
        for start in range(0, len(words) - window + 1, stride_words):
            prompt_text = " ".join(words[start:start + prompt_words])
            cont_text = " ".join(words[start + prompt_words:start + window])
            if prompt_text.strip() and cont_text.strip():
                pairs.append((prompt_text, cont_text))
                if len(pairs) >= max_pairs:
                    pbar.set_postfix(pairs=len(pairs))
                    pbar.close()
                    return pairs
        pbar.set_postfix(pairs=len(pairs))
    pbar.close()
    return pairs


class PrefixContinuationDataset(Dataset):
    def __init__(self, pairs, max_src_len=768, max_tgt_len=192):
        self.pairs = pairs
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]
        src_ids = encode(src_text)[:self.max_src_len]
        tgt_ids = ([BOS_ID] + encode(tgt_text))[:self.max_tgt_len - 1] + [EOS_ID]
        return {"src": src_ids, "tgt": tgt_ids}


def collate_prefix_continuation(batch):
    """For the encoder-decoder architecture: separate padded src/tgt tensors."""
    max_src = max(len(b["src"]) for b in batch)
    max_tgt = max(len(b["tgt"]) for b in batch)
    src_batch, src_mask, tgt_in, tgt_out, tgt_mask = [], [], [], [], []
    for b in batch:
        s = b["src"] + [PAD_ID] * (max_src - len(b["src"]))
        sm = [1] * len(b["src"]) + [0] * (max_src - len(b["src"]))
        t = b["tgt"] + [PAD_ID] * (max_tgt - len(b["tgt"]))
        tm = [1] * len(b["tgt"]) + [0] * (max_tgt - len(b["tgt"]))
        src_batch.append(s)
        src_mask.append(sm)
        tgt_in.append(t[:-1])
        tgt_out.append(t[1:])
        tgt_mask.append(tm[1:])
    return {
        "src_ids": torch.tensor(src_batch, dtype=torch.long),
        "src_mask": torch.tensor(src_mask, dtype=torch.long),
        "tgt_in": torch.tensor(tgt_in, dtype=torch.long),
        "tgt_out": torch.tensor(tgt_out, dtype=torch.long),
        "tgt_mask": torch.tensor(tgt_mask, dtype=torch.long),
    }


def collate_hybrid(batch):
    """For the hybrid prefix-LM architecture: a single concatenated [prompt ++ continuation]
    sequence per example, plus the (batch-uniform, due to padding) prompt length."""
    max_src = max(len(b["src"]) for b in batch)
    max_tgt = max(len(b["tgt"]) for b in batch)
    input_batch = []
    for b in batch:
        s = b["src"] + [PAD_ID] * (max_src - len(b["src"]))
        t = b["tgt"] + [PAD_ID] * (max_tgt - len(b["tgt"]))
        input_batch.append(s + t)
    return {
        "input_ids": torch.tensor(input_batch, dtype=torch.long),
        "prompt_len": max_src,
    }

## 9. Shared building blocks

`RMSNorm`, rotary position embeddings (RoPE), and the `SwiGLU` gated feed-forward used by the
hybrid and lightweight architectures.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def build_rope_cache(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]


def apply_rope(q, k, cos, sin):
    q_ = (q * cos) + (rotate_half(q) * sin)
    k_ = (k * cos) + (rotate_half(k) * sin)
    return q_, k_


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

## 10. Architecture 1 -- Decoder-only Transformer

Classic GPT-style decoder: learned absolute position embeddings, pre-LN blocks with standard
multi-head causal self-attention, GELU MLP, weight-tied output head.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, max_len):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.register_buffer("mask", torch.tril(torch.ones(max_len, max_len)).bool(), persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~self.mask[:T, :T], float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class GPTBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class DecoderOnlyGPT(nn.Module):
    """Architecture 1: classic decoder-only Transformer LM."""

    def __init__(self, vocab_size, n_layer, n_embd, n_head, d_ff, max_len=2048):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(max_len, n_embd)
        self.blocks = nn.ModuleList([GPTBlock(n_embd, n_head, d_ff, max_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=PAD_ID,
            )
        return loss, logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.8, top_p=0.95):
        # Simplified, non-KV-cached generation (recomputes full forward pass each step).
        # Positions are re-based to 0 each call since context is re-clipped to max_len.
        self.eval()
        for _ in range(max_new_tokens):
            ids_cond = input_ids[:, -self.max_len:]
            _, logits = self(ids_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids

## 11. Architecture 2 -- Encoder-Decoder Transformer

T5-style: separate encoder (bidirectional) and decoder (causal self-attn + cross-attn) stacks,
using **relative position bias** instead of absolute position embeddings. Trained directly on
(prompt -> continuation) pairs, matching the Phase-1 task format.

In [ ]:
class RelativePositionBias(nn.Module):
    def __init__(self, n_head, num_buckets=32, max_distance=128, bidirectional=True):
        super().__init__()
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, n_head)

    @staticmethod
    def _bucket(rel_pos, bidirectional, num_buckets, max_distance):
        ret = 0
        n = -rel_pos
        if bidirectional:
            num_buckets //= 2
            ret += (n < 0).long() * num_buckets
            n = torch.abs(n)
        else:
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2
        is_small = n < max_exact
        val_large = max_exact + (
            torch.log(n.float() / max_exact + 1e-6) / math.log(max_distance / max_exact) * (num_buckets - max_exact)
        ).long()
        val_large = torch.clamp(val_large, max=num_buckets - 1)
        ret = ret + torch.where(is_small, n, val_large)
        return ret

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        rel_pos = k_pos - q_pos
        buckets = self._bucket(rel_pos, self.bidirectional, self.num_buckets, self.max_distance)
        values = self.embedding(buckets)                    # (q_len, k_len, n_head)
        return values.permute(2, 0, 1).unsqueeze(0)          # (1, n_head, q_len, k_len)


class RelPosAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x_q, x_kv, bias=None, causal=False, key_padding_mask=None):
        B, Tq, C = x_q.shape
        Tk = x_kv.shape[1]
        q = self.q_proj(x_q).view(B, Tq, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, Tk, self.n_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, Tk, self.n_head, self.d_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if bias is not None:
            att = att + bias
        if causal:
            causal_mask = torch.tril(torch.ones(Tq, Tk, device=x_q.device)).bool()
            att = att.masked_fill(~causal_mask, float("-inf"))
        if key_padding_mask is not None:
            pad_keep = key_padding_mask[:, None, None, :].bool()   # (B,1,1,Tk), 1 = keep
            att = att.masked_fill(~pad_keep, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, Tq, C)
        return self.o_proj(out)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = RelPosAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, bias, key_padding_mask):
        h = self.ln1(x)
        x = x + self.attn(h, h, bias=bias, causal=False, key_padding_mask=key_padding_mask)
        x = x + self.ff(self.ln2(x))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = RelPosAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = RelPosAttention(d_model, n_head)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, enc_out, self_bias, enc_padding_mask):
        h = self.ln1(x)
        x = x + self.self_attn(h, h, bias=self_bias, causal=True)
        x = x + self.cross_attn(self.ln2(x), enc_out, bias=None, causal=False, key_padding_mask=enc_padding_mask)
        x = x + self.ff(self.ln3(x))
        return x


class EncoderDecoderTransformer(nn.Module):
    """Architecture 2: T5-style encoder-decoder Transformer with relative position bias."""

    def __init__(self, vocab_size, n_enc_layer, n_dec_layer, d_model, n_head, d_ff):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.enc_layers = nn.ModuleList([EncoderLayer(d_model, n_head, d_ff) for _ in range(n_enc_layer)])
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, n_head, d_ff) for _ in range(n_dec_layer)])
        self.enc_rel_bias = RelativePositionBias(n_head, bidirectional=True)
        self.dec_rel_bias = RelativePositionBias(n_head, bidirectional=False)
        self.ln_enc_f = nn.LayerNorm(d_model)
        self.ln_dec_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def encode(self, src_ids, src_mask):
        x = self.tok_emb(src_ids)
        bias = self.enc_rel_bias(src_ids.size(1), src_ids.size(1), src_ids.device)
        for layer in self.enc_layers:
            x = layer(x, bias, src_mask)
        return self.ln_enc_f(x)

    def forward(self, src_ids, src_mask, tgt_in, tgt_out=None):
        enc_out = self.encode(src_ids, src_mask)
        x = self.tok_emb(tgt_in)
        bias = self.dec_rel_bias(tgt_in.size(1), tgt_in.size(1), tgt_in.device)
        for layer in self.dec_layers:
            x = layer(x, enc_out, bias, src_mask)
        x = self.ln_dec_f(x)
        logits = self.head(x)
        loss = None
        if tgt_out is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1), ignore_index=PAD_ID)
        return loss, logits

    @torch.no_grad()
    def generate(self, src_ids, src_mask, max_new_tokens, bos_id, eos_id, temperature=0.8, top_p=0.95):
        # Simplified, non-KV-cached autoregressive decoding.
        self.eval()
        enc_out = self.encode(src_ids, src_mask)
        B = src_ids.size(0)
        tgt = torch.full((B, 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            x = self.tok_emb(tgt)
            bias = self.dec_rel_bias(tgt.size(1), tgt.size(1), tgt.device)
            h = x
            for layer in self.dec_layers:
                h = layer(h, enc_out, bias, src_mask)
            h = self.ln_dec_f(h)
            logits = self.head(h)[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            tgt = torch.cat([tgt, next_id], dim=1)
        return tgt

## 12. Architecture 3 -- Encoder/Hybrid (UniLM-style Prefix-LM)

A **single** Transformer stack (unlike Architecture 2's separate encoder/decoder), using RoPE +
RMSNorm + SwiGLU, with a UniLM-style attention mask: prompt tokens attend to each other
**bidirectionally**, continuation tokens attend **causally** to the prompt and to earlier
continuation tokens. This lets one stack behave like an encoder over the prompt and a decoder
over the continuation simultaneously.

In [ ]:
class RoPEAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, attn_mask, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~attn_mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class HybridBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = RoPEAttention(d_model, n_head)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff)

    def forward(self, x, attn_mask, cos, sin):
        x = x + self.attn(self.norm1(x), attn_mask, cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


def build_prefix_lm_mask(prompt_len, total_len, device):
    """prompt<->prompt: bidirectional. continuation->prompt: visible.
    continuation<->continuation: causal."""
    mask = torch.zeros(total_len, total_len, dtype=torch.bool, device=device)
    mask[:prompt_len, :prompt_len] = True
    mask[prompt_len:, :prompt_len] = True
    cont_len = total_len - prompt_len
    if cont_len > 0:
        mask[prompt_len:, prompt_len:] = torch.tril(torch.ones(cont_len, cont_len, device=device)).bool()
    return mask


class HybridPrefixLM(nn.Module):
    """Architecture 3: single-stack encoder/hybrid LM (RoPE, RMSNorm, SwiGLU,
    UniLM-style prefix-bidirectional + causal-continuation attention mask)."""

    def __init__(self, vocab_size, n_layer, d_model, n_head, d_ff, max_len=1536):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([HybridBlock(d_model, n_head, d_ff) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, prompt_len, labels=None):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        cos, sin = build_rope_cache(T, self.d_model // self.n_head, input_ids.device)
        mask = build_prefix_lm_mask(prompt_len, T, input_ids.device).unsqueeze(0).unsqueeze(0)
        for blk in self.blocks:
            x = blk(x, mask, cos, sin)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=PAD_ID,
            )
        return loss, logits

    @torch.no_grad()
    def generate(self, prompt_ids, max_new_tokens, temperature=0.8, top_p=0.95):
        self.eval()
        input_ids = prompt_ids
        prompt_len = prompt_ids.size(1)
        for _ in range(max_new_tokens):
            ids_cond = input_ids[:, -self.max_len:]
            cur_prompt_len = min(prompt_len, ids_cond.size(1))
            _, logits = self(ids_cond, cur_prompt_len)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids

## 13. Architecture 4 -- Lightweight / Efficient Transformer

A Llama-style efficient decoder: RoPE, RMSNorm, SwiGLU, and **Grouped-Query Attention** (far
fewer key/value heads than query heads), which cuts attention parameter count and KV-cache
memory relative to Architecture 1 -- the explicit "efficiency" lever for this architecture
type.

In [ ]:
class GQAAttention(nn.Module):
    """Grouped-Query Attention: n_kv_head < n_head, reducing parameters and
    (at inference, with KV caching) memory -- the key efficiency trick here."""

    def __init__(self, d_model, n_head, n_kv_head):
        super().__init__()
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.d_head = d_model // n_head
        self.q_proj = nn.Linear(d_model, n_head * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_head * self.d_head, d_model, bias=False)

    def forward(self, x, causal_mask, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~causal_mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)


class LightweightBlock(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, d_ff):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_head, n_kv_head)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff)

    def forward(self, x, mask, cos, sin):
        x = x + self.attn(self.norm1(x), mask, cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


class LightweightLlama(nn.Module):
    """Architecture 4: efficient/lightweight decoder-only Transformer
    (RoPE, RMSNorm, SwiGLU, Grouped-Query Attention)."""

    def __init__(self, vocab_size, n_layer, d_model, n_head, n_kv_head, d_ff, max_len=2048):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([LightweightBlock(d_model, n_head, n_kv_head, d_ff) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        cos, sin = build_rope_cache(T, self.d_model // self.n_head, input_ids.device)
        mask = torch.tril(torch.ones(T, T, device=input_ids.device)).bool().unsqueeze(0).unsqueeze(0)
        for blk in self.blocks:
            x = blk(x, mask, cos, sin)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=PAD_ID,
            )
        return loss, logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.8, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            ids_cond = input_ids[:, -self.max_len:]
            _, logits = self(ids_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids

## 14. Build the selected model

In [ ]:
def build_model():
    cfg = MODEL_CONFIGS[ARCHITECTURE]
    if ARCHITECTURE == "decoder_only":
        m = DecoderOnlyGPT(VOCAB_SIZE_ACTUAL, cfg["n_layer"], cfg["n_embd"], cfg["n_head"], cfg["d_ff"],
                            max_len=BLOCK_SIZE)
    elif ARCHITECTURE == "encoder_decoder":
        m = EncoderDecoderTransformer(VOCAB_SIZE_ACTUAL, cfg["n_enc_layer"], cfg["n_dec_layer"],
                                       cfg["d_model"], cfg["n_head"], cfg["d_ff"])
    elif ARCHITECTURE == "hybrid_prefixlm":
        m = HybridPrefixLM(VOCAB_SIZE_ACTUAL, cfg["n_layer"], cfg["d_model"], cfg["n_head"], cfg["d_ff"],
                            max_len=1152)  # generous headroom over the 768-token prompt + 220-token generation used below
    elif ARCHITECTURE == "lightweight":
        m = LightweightLlama(VOCAB_SIZE_ACTUAL, cfg["n_layer"], cfg["d_model"], cfg["n_head"],
                              cfg["n_kv_head"], cfg["d_ff"], max_len=BLOCK_SIZE)
    else:
        raise ValueError(ARCHITECTURE)

    n_params = sum(p.numel() for p in m.parameters())
    print(f"{ARCHITECTURE}: {n_params / 1e6:.2f}M parameters")
    assert 50e6 <= n_params <= 300e6, (
        f"Model has {n_params/1e6:.1f}M parameters -- competition requires 50M-300M "
        f"(Sec. 7.5). Adjust MODEL_CONFIGS['{ARCHITECTURE}']."
    )
    return m.to(DEVICE)


model = build_model()

## 15. Build the training/validation dataloaders for the selected architecture

In [ ]:
if ARCHITECTURE in ("decoder_only", "lightweight"):
    print("Building contiguous token-block dataset for causal-LM training...")
    all_tokens = build_token_stream(raw_documents, TRAIN_TOKENS + VAL_TOKENS)
    split_idx = int(len(all_tokens) * (TRAIN_TOKENS / (TRAIN_TOKENS + VAL_TOKENS)))
    train_tokens, val_tokens = all_tokens[:split_idx], all_tokens[split_idx:]
    train_ds = BlockLMDataset(make_blocks(train_tokens, BLOCK_SIZE))
    val_ds = BlockLMDataset(make_blocks(val_tokens, BLOCK_SIZE))
    print(f"Train blocks: {len(train_ds)} | Val blocks: {len(val_ds)}")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               collate_fn=collate_blocks, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             collate_fn=collate_blocks, drop_last=True)

else:
    print("Building prefix/continuation pair dataset...")
    n_val_docs = max(1, int(0.02 * len(raw_documents)))
    val_docs, train_docs = raw_documents[:n_val_docs], raw_documents[n_val_docs:]
    train_pairs = build_prefix_continuation_pairs(train_docs, max_pairs=150000)
    val_pairs = build_prefix_continuation_pairs(val_docs, max_pairs=3000)
    print(f"Train pairs: {len(train_pairs)} | Val pairs: {len(val_pairs)}")

    train_ds = PrefixContinuationDataset(train_pairs)
    val_ds = PrefixContinuationDataset(val_pairs)
    collate_fn = collate_prefix_continuation if ARCHITECTURE == "encoder_decoder" else collate_hybrid

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               collate_fn=collate_fn, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             collate_fn=collate_fn, drop_last=True)

## 16. Training loop

A single generic loop that dispatches to the right forward-pass signature depending on
`ARCHITECTURE`. Uses AdamW, cosine LR schedule with warmup, gradient accumulation, and mixed
precision.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))

total_steps = MAX_STEPS or max(1, (len(train_loader) * EPOCHS) // GRAD_ACCUM_STEPS)


def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))


scheduler = LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


def run_batch(batch):
    if ARCHITECTURE in ("decoder_only", "lightweight"):
        input_ids = batch["input_ids"].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            loss, _ = model(input_ids, labels=input_ids)
        return loss
    elif ARCHITECTURE == "encoder_decoder":
        src_ids = batch["src_ids"].to(DEVICE)
        src_mask = batch["src_mask"].to(DEVICE)
        tgt_in = batch["tgt_in"].to(DEVICE)
        tgt_out = batch["tgt_out"].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            loss, _ = model(src_ids, src_mask, tgt_in, tgt_out)
        return loss
    else:  # hybrid_prefixlm
        input_ids = batch["input_ids"].to(DEVICE)
        prompt_len = batch["prompt_len"]
        labels = input_ids.clone()
        labels[:, :prompt_len] = PAD_ID   # prompt tokens see bidirectional context -> exclude from LM loss
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            loss, _ = model(input_ids, prompt_len, labels=labels)
        return loss


@torch.no_grad()
def evaluate(loader, max_batches=50):
    model.eval()
    total_loss, n = 0.0, 0
    n_batches = min(max_batches, len(loader))
    eval_bar = tqdm(loader, total=n_batches, desc="eval", unit="batch", leave=False)
    for i, batch in enumerate(eval_bar):
        if i >= max_batches:
            break
        batch_loss = run_batch(batch).item()
        total_loss += batch_loss
        n += 1
        eval_bar.set_postfix(loss=f"{batch_loss:.4f}")
    eval_bar.close()
    model.train()
    return total_loss / max(1, n)


history = {"step": [], "train_loss": [], "eval_step": [], "eval_loss": []}
step = 0
model.train()
start_time = time.time()
optimizer.zero_grad()

print(f"Starting training | target optimizer steps ~ {total_steps}")
stop_training = False
# Live progress bar across optimizer steps -- the key thing to watch during a Kaggle/Colab run.
progress_bar = tqdm(total=total_steps, desc=f"training ({ARCHITECTURE})", unit="step")

for epoch in range(EPOCHS):
    if stop_training:
        break
    for i, batch in enumerate(train_loader):
        loss = run_batch(batch) / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()

        if (i + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step += 1
            progress_bar.update(1)
            progress_bar.set_postfix(
                loss=f"{loss.item() * GRAD_ACCUM_STEPS:.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}",
            )

            if step % LOG_EVERY == 0:
                elapsed_min = (time.time() - start_time) / 60
                print(f"step {step}/{total_steps} | loss {loss.item() * GRAD_ACCUM_STEPS:.4f} "
                      f"| lr {scheduler.get_last_lr()[0]:.2e} | {elapsed_min:.1f} min")
                history["step"].append(step)
                history["train_loss"].append(loss.item() * GRAD_ACCUM_STEPS)

            if step % EVAL_EVERY == 0:
                val_loss = evaluate(val_loader)
                ppl = math.exp(min(val_loss, 20))
                print(f"  [eval] step {step} | val_loss {val_loss:.4f} | val_ppl {ppl:.2f}")
                history["eval_step"].append(step)
                history["eval_loss"].append(val_loss)
                progress_bar.set_postfix(
                    loss=f"{loss.item() * GRAD_ACCUM_STEPS:.4f}",
                    lr=f"{scheduler.get_last_lr()[0]:.2e}",
                    val_ppl=f"{ppl:.2f}",
                )

            if step % SAVE_EVERY == 0:
                torch.save(model.state_dict(), os.path.join(CKPT_DIR, f"step_{step}.pt"))

            if MAX_STEPS is not None and step >= MAX_STEPS:
                stop_training = True
                break

progress_bar.close()
print(f"Training finished in {(time.time() - start_time) / 3600:.2f} hours, {step} optimizer steps.")
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "final_model.pt"))
print("Final model weights saved to:", os.path.join(OUTPUT_DIR, "final_model.pt"))

## 17. Plot the training/validation loss curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(history["step"], history["train_loss"], label="train loss")
plt.plot(history["eval_step"], history["eval_loss"], label="val loss")
plt.xlabel("optimizer step")
plt.ylabel("loss")
plt.title(f"Phase 1 ({ARCHITECTURE}) training/validation loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curve.png"), dpi=150)
plt.show()
print("Final validation perplexity:", math.exp(min(history["eval_loss"][-1], 20)) if history["eval_loss"] else "n/a")

## 18. Load the official Phase-1 test prompts

In [ ]:
test_path, used_test_filename = download_first_available(
    TEST_FILE_CANDIDATES,
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
)
print("Using test file:", used_test_filename)

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test prompts:", len(test_data))

## 19. Generation functions (one per architecture family)

In [ ]:
def generate_continuation_causal(prompt_text, target_words=CONT_WORDS):
    """decoder_only / lightweight: feed the prompt, sample autoregressively."""
    prompt_ids = encode(prompt_text)
    max_prompt_tokens = model.max_len - 240
    prompt_ids = prompt_ids[-max_prompt_tokens:]
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    out_ids = model.generate(input_ids, max_new_tokens=220, temperature=0.8, top_p=0.95)
    gen_ids = out_ids[0, len(prompt_ids):].tolist()
    words = decode(gen_ids).split()[:target_words]
    return " ".join(words)


def generate_continuation_seq2seq(prompt_text, target_words=CONT_WORDS):
    """encoder_decoder: encode the prompt, decode the continuation from <bos>."""
    src_ids = encode(prompt_text)[:768]
    src = torch.tensor([src_ids], dtype=torch.long, device=DEVICE)
    src_mask = torch.ones_like(src)
    out = model.generate(src, src_mask, max_new_tokens=220, bos_id=BOS_ID, eos_id=EOS_ID,
                          temperature=0.8, top_p=0.95)
    gen_ids = out[0, 1:].tolist()  # drop <bos>
    if EOS_ID in gen_ids:
        gen_ids = gen_ids[:gen_ids.index(EOS_ID)]
    words = decode(gen_ids).split()[:target_words]
    return " ".join(words)


def generate_continuation_hybrid(prompt_text, target_words=CONT_WORDS):
    """hybrid_prefixlm: prompt is the bidirectional prefix, continuation is generated causally."""
    prompt_ids = encode(prompt_text)[:768]
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    out_ids = model.generate(input_ids, max_new_tokens=220, temperature=0.8, top_p=0.95)
    gen_ids = out_ids[0, len(prompt_ids):].tolist()
    words = decode(gen_ids).split()[:target_words]
    return " ".join(words)


GEN_FN = {
    "decoder_only": generate_continuation_causal,
    "lightweight": generate_continuation_causal,
    "encoder_decoder": generate_continuation_seq2seq,
    "hybrid_prefixlm": generate_continuation_hybrid,
}[ARCHITECTURE]

## 20. Generate predictions for every official test prompt

In [ ]:
model.eval()
predictions = {}
items = list(test_data.items())
print(f"Generating continuations for {len(items)} test prompts using '{ARCHITECTURE}'...")

for ex_id, record in tqdm(items, desc="generating predictions", unit="prompt"):
    prompt_text = record["text"]
    gen_text = GEN_FN(prompt_text)
    predictions[str(ex_id)] = {"text": gen_text}

## 21. Save `prediction.json` and build the Codabench `submission.zip`

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
print("Saved:", PREDICTION_PATH)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_PATH, arcname="prediction.json")
print("Codabench submission ready:", SUBMISSION_ZIP)

print()
print(f">>> This submission used ARCHITECTURE = '{ARCHITECTURE}'.")
print(">>> Change ARCHITECTURE in the CONFIG cell (Section 5) and re-run the notebook")
print(">>> to produce your other required Phase-1 architecture submissions.")

## 22. (Recommended) Local evaluation before you submit

The competition provides no public Phase-1 dev set, so Sec. 16 of the capstone doc asks you to
build your own held-out labelled split. This reserves some documents, treats a later span of
each as a synthetic "reference continuation", and scores your model's generations against it
with the same metric family used on Codabench (BERTScore / ROUGE-L / BLEU-4, Sec. 7.8).

In [ ]:
def build_local_eval_set(documents, n_examples=100, prompt_words=PROMPT_WORDS, cont_words=CONT_WORDS):
    examples = []
    for doc in documents:
        words = doc["text"].split()
        if len(words) < prompt_words + cont_words:
            continue
        examples.append({
            "prompt": " ".join(words[:prompt_words]),
            "reference": " ".join(words[prompt_words:prompt_words + cont_words]),
        })
        if len(examples) >= n_examples:
            break
    return examples


local_eval_examples = build_local_eval_set(raw_documents[-300:])
print("Local eval examples:", len(local_eval_examples))

hyps = [GEN_FN(ex["prompt"]) for ex in tqdm(local_eval_examples, desc="local eval generation", unit="example")]
refs = [ex["reference"] for ex in local_eval_examples]

import evaluate

bertscore_metric = evaluate.load("bertscore")
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

bs = bertscore_metric.compute(predictions=hyps, references=refs, lang="en")
rg = rouge_metric.compute(predictions=hyps, references=refs)
bl = bleu_metric.compute(predictions=hyps, references=[[r] for r in refs])

print(f"BERTScore F1 (mean): {sum(bs['f1']) / len(bs['f1']):.4f}")
print(f"ROUGE-L: {rg['rougeL']:.4f}")
print(f"BLEU-4: {bl['bleu']:.4f}")

## 23. Submission checklist (Sec. 19)

- [ ] `submission.zip` has `prediction.json` directly at its root (no extra folder) -- `zf.write(..., arcname="prediction.json")` above already guarantees this.
- [ ] Every official test ID from `test_data` is present in `predictions`, and no extra IDs were added (the generation loop above iterates `test_data.items()` exhaustively, so this holds by construction).
- [ ] Each `text` field contains **only** the generated continuation, not the original prompt.
- [ ] You trained **4 submissions across all 4 required architectures** (`decoder_only`, `encoder_decoder`, `hybrid_prefixlm`, `lightweight`) -- submissions from the unmodified starter code do not count toward the minimum (Sec. 7.7), but this notebook is your own implementation, so it does count once you've actually run training and have a real checkpoint behind each submission.
- [ ] You kept notes on what you varied per run (data mix, vocab size, context length, etc. -- Sec. 7.4) for your final report.

**On LLM usage disclosure:** the capstone's LLM-use rule (Sec. 1.9 / 17.5) allows using an LLM
to write the model architecture code and to understand it, but *not* for hyperparameter tuning
or result analysis/conclusions. If you use this notebook as your base, say so in your mandatory
LLM Declaration, and make sure any hyperparameter choices, architecture-comparison conclusions,
and error analysis in your report are **your own**, backed by the experiments you actually ran.